# Reversal MM Command Notebook

This notebook runs the cleaned logistic-only train / validation / test pipeline and prints only the core outputs:
- fill-surface OLS coefficients and $R^2$
- logistic train / validation / test classifier metrics
- validation balanced-inventory HftBacktest threshold sweep
- test balanced-inventory HftBacktest threshold sweep using the refit train+validation model
- test balanced-inventory HftBacktest results at the selected threshold using the refit train+validation model
- logistic regression coefficients and transform summary

In [1]:
import importlib
from pathlib import Path
import pandas as pd
from IPython.display import display

import reversal_mm_config as _cfg
import reversal_mm_pipeline as _pipe

_cfg = importlib.reload(_cfg)
_pipe = importlib.reload(_pipe)

PROJECT_ROOT = Path(_cfg.__file__).resolve().parent
pipeline_cfg, market_cfg, hbt_cfg, feature_cfg, model_cfg, strategy_cfg = _cfg.default_configs(PROJECT_ROOT)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /Users/lib/Desktop/valuation/during /RSM MSBA/github/Crpto_Strategy


In [ ]:
results = results = _pipe.run_full_pipeline(
    pipeline_cfg,
    market_cfg,
    hbt_cfg,
    feature_cfg,
    model_cfg,
    strategy_cfg,
    start_window=28,
)

print(_pipe.build_console_report(results))

Running sliding-window trial across 79 day files with 51 windows (21 train / 7 validation / 1 test).
Selected window range: 28 -> 51 (24 windows).
Loaded 27 preserved summary rows from sliding_window_summary.csv.

[Window 1/24 | global 28/51] train 2026-01-28 -> 2026-02-17, validation 2026-02-18 -> 2026-02-24, test 2026-02-25 -> 2026-02-25
Preparing window data, continuous state, snapshots, and sampled orders...
Validating input day files...
Loading/building continuous market state across all configured day files...
Mapping configured day windows into the continuous state...
Loading/sampling continuous maker-order stream across all configured day files...
Preparing end-of-day snapshots for subset backtests...
Building train-only fill surface...
Building train / validation / test datasets...


In [ ]:
print('=== Fill Surface OLS Summary ===')
display(pd.DataFrame([results.fill_surface_ols['summary']]))
display(results.fill_surface_ols['coef_table'])

print('=== Logistic Train / Validation / Test Metrics ===')
display(results.logistic_metrics)

print('=== Validation Balanced-Inventory Backtest (Logistic) ===')
display(results.logistic_validation_backtest)

print('=== Test Balanced-Inventory Threshold Sweep (Logistic, refit on train+validation) ===')
display(results.logistic_test_threshold_backtest)

print('=== Logistic Transform Summary ===')
display(pd.DataFrame([{
    'selected_threshold': results.logistic_selected_threshold,
    'n_log1p_features': len(results.training.summary.get('log1p_features', [])),
    'n_interactions': len(results.training.summary.get('interaction_pairs', [])),
}]))
display(pd.DataFrame({'log1p_feature': results.training.summary.get('log1p_features', [])}))
display(pd.DataFrame(results.training.summary.get('interaction_pairs', []), columns=['left_feature', 'right_feature']))

print('=== Logistic Coefficients (refit on train+validation) ===')
display(results.training.logistic_coefficients)

print('=== Test Balanced-Inventory Backtest (Logistic, selected threshold, refit on train+validation) ===')
display(results.logistic_test_backtest)

In [ ]:
# import sys
# import importlib

# _prev_argv = sys.argv[:]
# try:
#     sys.argv = ["backtest.py"]
#     import backtest
#     importlib.reload(backtest)
#     rc = backtest.main()
# finally:
#     sys.argv = _prev_argv

# print("exit code:", rc)